# Assignment 2A — Prompting as System Design
## Prompt patterns, reasoning chains and query routing

**Group 172**

| # | Name | Email | Student ID | Contribution |
|---|---|---|---|---|
| 1 | Naveen K. B | 2025af05145@wilp.bits-pilani.ac.in | 2025af05145 | 100% |
| 2 | Naman Aryan | 2025ae05277@wilp.bits-pilani.ac.in | 2025ae05277 | 100% |
| 3 | Patil Manaliben Dipakbhai | 2025ae05698@wilp.bits-pilani.ac.in | 2025ae05698 | 100% |
| 4 | Yeldona Desai V Vittal | 2025ae05316@wilp.bits-pilani.ac.in | 2025ae05316 | 100% |

(Names, emails, student IDs, and order copied verbatim from the team-details table in Assignment 1A's own submitted notebook.)

**Project summary**

| Item | Choice |
|---|---|
| Builds on | Assignment 1A (Group 172) -- domain corpus, instruction dataset, base model, and CPT + QLoRA pipeline, reused and cross-checked cell by cell against 1A's own submitted notebook (`Group172_Assignment1A_submission/LLM_Assignment_1A_Group_172.ipynb`) |
| Domain | Scientific Research Papers: deep learning and large language model research (same as 1A) |
| Source corpus | 13 open-access arXiv papers (same as 1A): BERT, GPT-3, LLaMA, Chinchilla, Transformer-XL, QLoRA, Prefix-Tuning, Prompt Tuning, Adapters, FlashAttention, DPO, InstructGPT, RAG |
| Instruction dataset | 65 pairs (5 per source paper), copied verbatim from 1A's own submission |
| Base model | `HuggingFaceTB/SmolLM2-360M` + CPT + QLoRA Adapter (r=16, alpha=32, target modules `q_proj`/`v_proj`) -- 1A's own adapter weights were lost (ephemeral Colab storage), so `train_adapter.py` regenerates them here following 1A's code exactly, including its chat-template format (a system prompt plus `### Question:` / `### Answer:` markers) and genuine 4-bit QLoRA SFT via a response-only loss mask |
| Generator settings | Greedy decoding, `max_new_tokens=200` everywhere -- matches the brief's fixed Setup instruction exactly. `repetition_penalty`/`no_repeat_ngram_size` were tried and then reverted after a real run showed they measurably hurt this small, lightly fine-tuned model (overall avg judge score 0.058 vs. 0.101 without them) by pushing it off its most-confident (if repetitive) continuations into less coherent ones |
| Judge model | `Qwen/Qwen2.5-7B-Instruct`, deterministic decoding |
| Benchmark | 10 queries from `instruction_dataset.jsonl`, stratified 5 factual / 3 procedural / 2 comparative, seed=42 |
| Prompting approach | Every pattern is a plain Python function using only `transformers` and `peft` -- no LangChain or other prompting frameworks, so every prompt sent to the model is visible in the code |
| Compute | Any CUDA GPU, detected at run time -- validated at reduced (dev) scale on a local CPU Mac; the full-fidelity run is on the BITS Prayogshala lab machine |

**Pipeline**
1. Setup: environment check, Colab bootstrap (no-op outside Colab), adapter retrain (skips automatically once the adapter is already on disk).
2. Step 1: dataset quality audit (length filter, question-type/per-paper coverage, template-repetition flag, verbatim-copy check, a 15-pair spot-check) and generator loading.
3. Step 2: the generator wrapper, the fixed 10-query benchmark, a smoke test.
4. Part A: five base prompting patterns (zero-shot, few-shot, instruction template, self-critique, structured output) benchmarked and scored with an LLM judge.
5. Part B: two multi-step reasoning problems; chain-of-thought with self-consistency; ReAct with a calculator tool; a plan-solve-verify chain built from A2's self-critique machinery.
6. Part C: a zero-shot query classifier, a router built on it, and a cost-vs-quality comparison against always-zero-shot.
7. Deliverables: a final, grounded check of every file and artifact the assignment brief asks for.

**Runtime status:** this notebook was developed and validated on a local CPU machine at reduced (dev) scale; the full-fidelity retrain (4 CPT epochs, 8 SFT epochs, matching 1A's hyperparameters exactly) is run separately on the BITS Prayogshala lab machine, and the resulting adapter is substituted in before final submission.


## Setup — Colab bootstrap

Running this notebook via `colab.research.google.com/github/.../Assignment2A_Group172.ipynb` only loads this file — not the rest of the repo (`instruction_dataset.jsonl`, `domain_corpus/`, `benchmark_10.json`, `train_adapter.py`) that the later cells need on disk next to it. This cell detects Colab, clones the repo, and installs the GPU dependency set. On a local machine or a labs box where the repo is already cloned and the environment already set up (per `RUNBOOK.md`), it does nothing.


In [1]:
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/VijayYD/2025AE05316-LLM-2A.git"
    REPO_DIR = "2025AE05316-LLM-2A"
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(REPO_DIR)
    print("Installing dependencies from requirements-gpu.txt ...")
    get_ipython().system(f"{sys.executable} -m pip install -q -r requirements-gpu.txt")
    print("Colab bootstrap done. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- skipping clone/install (assumes this notebook's own directory already has the repo files).")


Not running in Colab -- skipping clone/install (assumes this notebook's own directory already has the repo files).


## Setup — Retrain the adapter for this run

The dev-scale placeholder weights are intentionally **not** committed to the repo (they are regenerated fresh, and the full-fidelity CPT weights alone are ~1.3GB, over GitHub's per-file limit). This cell checks for a CUDA GPU:

- **GPU present** (labs, Colab T4+): runs `train_adapter.py --mode full`, reproducing Assignment 1A's exact two-stage pipeline (10 CPT epochs, 8 SFT epochs on the full dataset) — this is the run that should back the real submission numbers.
- **No GPU** (this local Mac): runs `train_adapter.py --mode dev`, a fast truncated run used only to validate that the rest of this notebook's pipeline runs end to end. Any generations produced under the dev-scale adapter are expected to be degenerate/undertrained and are documented as such wherever they appear below.

**Note on this run:** an earlier version of the cell below used a bare `!python` shell call, which does not necessarily resolve to this kernel's own interpreter; on the labs machine it silently failed with `ModuleNotFoundError: No module named 'transformers'`. The full-fidelity adapter actually loaded further below was trained separately, by hand, in a terminal (`python train_adapter.py --mode full`), confirmed complete before this notebook was run — so Part A's results are grounded in the real adapter despite that cell's failure. The cell is now fixed (uses `sys.executable` explicitly) and its output cleared; it has not been re-executed since the fix because the adapter already on disk is correct and a re-run would just repeat the full retrain unnecessarily.

**Update:** the cell now also checks whether the adapter files are already on disk and skips retraining entirely if so -- useful since this notebook has been re-run several times while iterating on Parts B/C, and redoing a full retrain each time wastes real labs time for no reason. Delete the adapter directories to force a fresh retrain.


In [2]:
import os
import sys
import torch

# NOTE: a prior run of this cell used the bare "!python" shell call, which does not
# necessarily resolve to this kernel's own interpreter (it depends on PATH in the
# subshell, not the kernel's sys.executable) -- on the labs machine this silently
# failed with "ModuleNotFoundError: No module named 'transformers'" while the rest
# of the notebook kept running against an adapter trained separately, by hand, in a
# terminal. Using sys.executable explicitly fixes that.

ADAPTER_FILES = ["cpt_final_model/model.safetensors", "final_qlora_adapter/adapter_model.safetensors"]
adapter_already_trained = all(os.path.exists(p) for p in ADAPTER_FILES)

if adapter_already_trained:
    print("Adapter files already on disk:")
    for p in ADAPTER_FILES:
        print(f"  {p}")
    print("Skipping retrain. Delete these (or the cpt_final_model/final_qlora_adapter directories) "
          "and re-run this cell to force a fresh retrain.")
    RETRAIN_MODE = "skipped (adapter already on disk)"
elif torch.cuda.is_available():
    print(f"CUDA GPU detected ({torch.cuda.get_device_name(0)}) -- running the full-fidelity retrain.")
    print(f"{sys.executable} train_adapter.py --mode full")
    get_ipython().system(f"{sys.executable} train_adapter.py --mode full")
    RETRAIN_MODE = "full"
else:
    print("No CUDA GPU detected -- running the fast dev-scale retrain for local pipeline validation only.")
    print(f"{sys.executable} train_adapter.py --mode dev")
    get_ipython().system(f"{sys.executable} train_adapter.py --mode dev")
    RETRAIN_MODE = "dev"

print("\nRETRAIN_MODE =", RETRAIN_MODE)


Adapter files already on disk:
  cpt_final_model/model.safetensors
  final_qlora_adapter/adapter_model.safetensors
Skipping retrain. Delete these (or the cpt_final_model/final_qlora_adapter directories) and re-run this cell to force a fresh retrain.

RETRAIN_MODE = skipped (adapter already on disk)


## Step 0 — Environment check

In [3]:
import sys, platform
import torch
import transformers
import peft

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Selected device:", DEVICE)


Python: 3.11.11
Platform: Linux-6.8.0-117-generic-x86_64-with-glibc2.35
torch: 2.5.1+cu121
transformers: 4.46.3
peft: 0.13.2
CUDA available: True
MPS available: False
Selected device: cuda


## Step 1 — Dataset

`instruction_dataset.jsonl` (65 pairs): the real Assignment 1A instruction dataset, copied verbatim from 1A's own submitted notebook -- 5 hand-written pairs per source paper across all 13 papers, each tagged with its source document and question type (factual/procedural/comparative). An earlier version of this project used a different, invented 58-pair dataset built against the wrong 1A submission path; it has been replaced with the real one. Audited below against the quality rules: minimum pair count, a response-length filter (>= 30 words), question-type and per-paper coverage (using the dataset's own ground-truth fields), template-repetition flagging (> 20% of pairs), and a verbatim-copy check against the source papers.

In [4]:
import json
import re
from collections import Counter

with open("instruction_dataset.jsonl") as f:
    dataset_pairs = [json.loads(l) for l in f]

print(f"Total pairs: {len(dataset_pairs)} (minimum required: 40-50)")
print(f"Field names: {sorted(dataset_pairs[0].keys())}")


Total pairs: 65 (minimum required: 40-50)
Field names: ['instruction', 'question_type', 'response', 'source']


### Response-length filter (>= 30 words)


In [5]:
def word_count(s):
    return len(s.split())

lengths = [word_count(p["response"]) for p in dataset_pairs]
short_pairs = [(i, n) for i, n in enumerate(lengths) if n < 30]

print(f"Pairs failing the >= 30-word filter: {len(short_pairs)}/{len(dataset_pairs)}")
print(f"Response word count: min={min(lengths)} max={max(lengths)} mean={sum(lengths)/len(lengths):.1f}")
print("\nNote: most failures are among the original 50 (1A) pairs, kept unchanged for retraining "
      "fidelity per the earlier approved decision; the 8 supplementary pairs were written to pass "
      "this filter.")


Pairs failing the >= 30-word filter: 0/65
Response word count: min=38 max=54 mean=45.6

Note: most failures are among the original 50 (1A) pairs, kept unchanged for retraining fidelity per the earlier approved decision; the 8 supplementary pairs were written to pass this filter.


### Question-type and per-paper coverage


In [6]:
# The real 1A dataset carries its own ground-truth "question_type" and "source" fields
# (each pair was hand-tagged when written) -- using them directly is more accurate than
# guessing a question's type or source paper from keywords.
types = [p["question_type"] for p in dataset_pairs]
print("Question-type distribution:", Counter(types))

sources = [p["source"] for p in dataset_pairs]
source_counts = Counter(sources)
print("\nPer-paper coverage:", dict(sorted(source_counts.items())))
print(f"Papers covered: {len(source_counts)} of 13")


Question-type distribution: Counter({'factual': 38, 'procedural': 14, 'comparative': 13})

Per-paper coverage: {'adapters': 5, 'bert': 5, 'chinchilla': 5, 'dpo': 5, 'flash_attention': 5, 'gpt3': 5, 'instructgpt': 5, 'llama': 5, 'prefix_tuning': 5, 'prompt_tuning': 5, 'qlora': 5, 'rag': 5, 'transformer_xl': 5}
Papers covered: 13 of 13


### Template-repetition flag (> 20% of pairs)


In [7]:
def leading_template(instruction):
    return " ".join(instruction.strip().split()[:3]).lower().rstrip("?,.")

templates = [leading_template(p["instruction"]) for p in dataset_pairs]
template_counts = Counter(templates)
top_template, top_count = template_counts.most_common(1)[0]
top_pct = 100 * top_count / len(dataset_pairs)

print(f"Most frequent leading phrase: {top_template!r} -- {top_count}/{len(dataset_pairs)} pairs ({top_pct:.1f}%)")
print("Flag (> 20%):", "YES" if top_pct > 20 else "no")
print("\nTop 5 leading phrases:")
for tmpl, cnt in template_counts.most_common(5):
    print(f"  {cnt:3d} ({100*cnt/len(dataset_pairs):4.1f}%)  {tmpl!r}")


Most frequent leading phrase: 'how does the' -- 4/65 pairs (6.2%)
Flag (> 20%): no

Top 5 leading phrases:
    4 ( 6.2%)  'how does the'
    4 ( 6.2%)  'what is the'
    3 ( 4.6%)  'what are the'
    3 ( 4.6%)  'how is the'
    2 ( 3.1%)  'why did the'


### Verbatim-copy check (>= 12 consecutive words shared with a source paper)


In [8]:
from pathlib import Path

corpus_dir = Path("domain_corpus")
corpus_texts = {f.stem: f.read_text(encoding="utf-8", errors="ignore") for f in corpus_dir.glob("*.txt")}

def normalize(text):
    return re.sub(r"\s+", " ", text.lower()).strip()

def ngrams(words, n=12):
    return set(tuple(words[i:i + n]) for i in range(len(words) - n + 1))

corpus_ngrams = {name: ngrams(normalize(text).split()) for name, text in corpus_texts.items()}

verbatim_hits = []
for i, p in enumerate(dataset_pairs):
    resp_words = normalize(p["response"]).split()
    if len(resp_words) < 12:
        continue
    resp_ngrams = ngrams(resp_words)
    for name, cng in corpus_ngrams.items():
        shared = resp_ngrams & cng
        if shared:
            verbatim_hits.append((i, name, len(shared)))

print(f"Responses with a >= 12-word verbatim span shared with a source paper: {len(verbatim_hits)}/{len(dataset_pairs)}")
for i, name, n in verbatim_hits[:10]:
    print(f"  idx {i} vs {name}: {n} shared 12-grams")


Responses with a >= 12-word verbatim span shared with a source paper: 0/65


### Spot-check sample (15 pairs, fixed seed)


In [9]:
import random

random.seed(42)
sample_idx = random.sample(range(len(dataset_pairs)), 15)

for i in sample_idx:
    print(f"\n[{i}] Q: {dataset_pairs[i]['instruction']}")
    print(f"     A: {dataset_pairs[i]['response']}")



[14] Q: How does the LLaMA tokenizer handle numbers and unknown characters?
     A: LLaMA tokenizes text with byte-pair encoding implemented in SentencePiece. Every number is split into individual digits, and any character missing from the vocabulary falls back to its UTF-8 bytes, so no input string is ever mapped to an unknown token.

[3] Q: Which corpora and model sizes were used for BERT pre-training?
     A: BERT was pre-trained on BooksCorpus, about 800 million words, and English Wikipedia, about 2,500 million words, keeping only text passages. Two sizes were released: BERT-Base with 12 layers and 110 million parameters, and BERT-Large with 24 layers and 340 million parameters.

[47] Q: How does FlashAttention handle the backward pass efficiently?
     A: Instead of storing the large attention matrix for the backward pass, FlashAttention keeps only the output and the softmax normalisation statistics. It then recomputes the attention blocks on chip during backpropagation, which ad

Vijay's spot-check verdict: reviewed all 15 sampled pairs above (real 1A dataset, 65 pairs). All are concise, accurate answers written in the dataset's own words -- none read as verbatim text copied from a source paper, and none are factually incorrect. No pairs flagged.


## Step 1 — Load the fine-tuned generator (1A pipeline reproduction)

The generator is `HuggingFaceTB/SmolLM2-360M` after Assignment 1A's two-stage pipeline: continual pre-training (CPT) on the domain corpus, then QLoRA instruction fine-tuning on `instruction_dataset.jsonl`.

**Deviation from 1A's literal code, and why:** 1A's `tokenize_sft` function set `labels = input_ids.copy()` without masking padding positions. Since prompts are padded to `max_length=512` while real content is far shorter, this computes cross-entropy loss mostly over padding tokens, pushing the loss toward `ln(vocab_size) ≈ 10.8` regardless of real learning and destabilizing training (verified: reproducing 1A's literal code on its own first training batch gives loss ≈10.8–11, not the ≈7.5 shown in 1A's stored notebook output — the two are inconsistent). The fix (masking padding positions with `-100` in the labels) was verified to restore a stable, sane loss trajectory and is applied in `train_adapter.py`. No other hyperparameters were changed.

**Scale note:** the adapter loaded below is from a *dev-scale* run (40 packed CPT sequences, 15 max steps per stage) used only to validate this pipeline end to end on a CPU machine. It is a placeholder — the full-fidelity adapter (1A's exact 10 CPT / 8 SFT epochs on the complete corpus and dataset) is trained separately on Prayogshala and substituted here before final submission, by pointing `CPT_MODEL_PATH` / `ADAPTER_PATH` at the new weights.


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
CPT_MODEL_PATH = "./cpt_final_model"
ADAPTER_PATH = "./final_qlora_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(CPT_MODEL_PATH, torch_dtype=torch.float32)
generator_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
generator_model.to(DEVICE)
generator_model.eval()

n_trainable = sum(p.numel() for p in generator_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in generator_model.parameters())
print(f"Loaded generator: {BASE_MODEL_ID} + CPT + QLoRA adapter")
print(f"Trainable params (should be 0 in eval/inference): {n_trainable} / {n_total}")


Loaded generator: HuggingFaceTB/SmolLM2-360M + CPT + QLoRA adapter
Trainable params (should be 0 in eval/inference): 0 / 363459520


## Step 2 — Generator wrapper, per-call helper, and the pattern-function contract

- `generate(prompt, **overrides)` — raw wrapper around the model: builds the 1A-style instruction prompt, runs greedy decoding with `max_new_tokens=200` by default, returns the decoded continuation text only.
- `call_generator(prompt, **overrides)` — per-call helper used by every prompt pattern: wraps `generate`, measuring prompt tokens, generated tokens, and wall-clock latency for that one call.
- **Pattern function contract:** every prompt pattern implemented in Parts A/B/C is a plain Python function `pattern(query) -> (answer, tokens_used, latency_sec)`, built on top of one or more `call_generator` calls. `tokens_used` is the sum of prompt + generated tokens across all model calls the pattern makes (multi-call patterns such as self-critique or ReAct make several), and `latency_sec` is the pattern's total wall-clock time.


In [11]:
import time

# repetition_penalty=1.3 + no_repeat_ngram_size=3 were tried after real runs showed this
# small, lightly fine-tuned model looping on the same phrase under greedy decoding. A real
# labs run with both settings active was measurably WORSE (overall avg judge score 0.058
# vs. 0.101 without them, nonzero scores 2/69 vs. 6/69): instead of reaching a cleaner
# conclusion, the model was pushed off its most-confident (if repetitive) continuations into
# much lower-probability, often bizarre completions (LaTeX-like notation, fabricated
# citations, invented jargon) -- a known failure mode when a repetition penalty is applied to
# a small, narrowly-tuned model that doesn't have many good alternative continuations once its
# top choice is suppressed. Reverted to plain greedy decoding, which also matches the brief's
# literal "Fix the decoding settings: greedy ... max_new_tokens = 200" instruction exactly.
GEN_DEFAULTS = dict(max_new_tokens=200, do_sample=False)

# Matches 1A's own SFT chat template exactly (verified cell-by-cell against 1A's submitted
# notebook) -- a system prompt followed by "### Question:" / "### Answer:" markers. The
# adapter was only ever trained on this format; the previous "### Instruction:"/"### Response:"
# template did not match what the model was actually fine-tuned on.
SYSTEM_PROMPT = (
    "You are a research assistant that answers questions about deep learning and "
    "large language model papers accurately and concisely."
)
PROMPT_TEMPLATE = "{system}\n\n### Question:\n{instruction}\n\n### Answer:\n"

def generate(prompt: str, **overrides) -> str:
    """Raw generation call: formats `prompt` using 1A's own chat format, greedily decodes, returns the continuation text (prompt stripped, stops at the first new '### Question:' if the model keeps going)."""
    settings = {**GEN_DEFAULTS, **overrides}
    full_prompt = PROMPT_TEMPLATE.format(system=SYSTEM_PROMPT, instruction=prompt)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **settings,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    text = text.split("### Question:")[0].strip()
    return text

def call_generator(prompt: str, **overrides):
    """Per-call helper: wraps generate(), returns (text, prompt_tokens, generated_tokens, latency_sec)."""
    prompt_tokens = len(tokenizer(PROMPT_TEMPLATE.format(system=SYSTEM_PROMPT, instruction=prompt), add_special_tokens=False)["input_ids"])
    t0 = time.perf_counter()
    text = generate(prompt, **overrides)
    latency_sec = time.perf_counter() - t0
    generated_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return text, prompt_tokens, generated_tokens, latency_sec

print("Generation defaults:", GEN_DEFAULTS)
print("Prompt format (matches 1A's SFT chat template exactly):")
print(PROMPT_TEMPLATE.format(system=SYSTEM_PROMPT, instruction="<query>"))
print("Pattern contract: pattern(query) -> (answer, tokens_used, latency_sec)")


Generation defaults: {'max_new_tokens': 200, 'do_sample': False}
Prompt format (matches 1A's SFT chat template exactly):
You are a research assistant that answers questions about deep learning and large language model papers accurately and concisely.

### Question:
<query>

### Answer:

Pattern contract: pattern(query) -> (answer, tokens_used, latency_sec)


## Step 2 — Fixed benchmark set

Loaded from `benchmark_10.json`: 10 queries drawn from the instruction dataset via stratified random sampling (seed=42, 5 factual / 3 procedural / 2 comparative), approved earlier. This fixed set is reused, unchanged, across every pattern in Parts A and B so results are comparable.


In [12]:
import json
import pandas as pd

with open("benchmark_10.json") as f:
    benchmark = json.load(f)

benchmark_df = pd.DataFrame(benchmark)
print(f"Benchmark size: {len(benchmark_df)}")
print(benchmark_df["type"].value_counts())
benchmark_df[["source_idx", "type", "instruction"]]


Benchmark size: 10
type
factual        5
procedural     3
comparative    2
Name: count, dtype: int64


,source_idx,type,instruction
0,27,factual,How much memory does double quantization save ...
1,30,factual,What is prefix-tuning and how many parameters ...
2,25,factual,What are the three main innovations introduced...
3,57,comparative,How does PPO-ptx differ from the plain PPO mod...
4,8,procedural,How did the GPT-3 authors check whether benchm...
5,4,procedural,Outline how BERT is adapted to a downstream cl...
6,14,factual,How does the LLaMA tokenizer handle numbers an...
7,51,procedural,How is the DPO training loss computed for a si...
8,3,factual,Which corpora and model sizes were used for BE...
9,53,comparative,How does DPO compare with PPO-based RLHF in th...


## Step 2 — Smoke test

One real call through the full wrapper (`call_generator`) on the first benchmark query, to confirm the generator, tokenizer, and timing/token accounting all work end to end before any pattern is built on top of them.


In [13]:
smoke_query = benchmark[0]["instruction"]
smoke_text, smoke_prompt_tok, smoke_gen_tok, smoke_latency = call_generator(smoke_query)

print("Query:", smoke_query)
print("\n--- Generated answer ---")
print(smoke_text)
print("\n--- Accounting ---")
print(f"Prompt tokens: {smoke_prompt_tok}")
print(f"Generated tokens: {smoke_gen_tok}")
print(f"Latency (sec): {smoke_latency:.3f}")


Query: How much memory does double quantization save in QLoRA?

--- Generated answer ---
The double quantization of the QLoRA model saves 1.5 GB of memory, which is equivalent to 100 GB of RAM.

--- Accounting ---
Prompt tokens: 46
Generated tokens: 31
Latency (sec): 3.953


## Part A — Base patterns

Five patterns are compared on the fixed 10-query benchmark: zero-shot (baseline), few-shot, instruction, self-critique, and structured output. Each is a plain Python function `pattern(query) -> (answer, tokens_used, latency_sec)` built on `call_generator` / `call_generator_raw`.


In [14]:
def generate_raw(full_prompt: str, stop_strings=None, **overrides) -> str:
    """Like generate(), but takes an already fully-formatted prompt (used by patterns that
    build multi-block prompts, e.g. few-shot). Truncates the decoded continuation at the first
    occurrence of any string in stop_strings, the same post-hoc truncation generate() uses."""
    settings = {**GEN_DEFAULTS, **overrides}
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **settings,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    for stop in (stop_strings or []):
        if stop in text:
            text = text.split(stop)[0]
    return text.strip()

def call_generator_raw(full_prompt: str, stop_strings=None, **overrides):
    """Per-call helper for generate_raw(): returns (text, prompt_tokens, generated_tokens, latency_sec)."""
    prompt_tokens = len(tokenizer(full_prompt, add_special_tokens=False)["input_ids"])
    t0 = time.perf_counter()
    text = generate_raw(full_prompt, stop_strings=stop_strings, **overrides)
    latency_sec = time.perf_counter() - t0
    generated_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return text, prompt_tokens, generated_tokens, latency_sec


### A0. Zero-shot baseline

Send the query as-is, in the training format. Not graded on its own; used as the comparison point for every other pattern.


In [15]:
def pattern_zero_shot(query):
    text, p_tok, g_tok, lat = call_generator(query)
    return text, p_tok + g_tok, lat


### A1. Few-shot

3 worked examples from `instruction_dataset.jsonl` (indices 1, 10, 47, re-selected after the dataset was replaced with the real 1A dataset), placed before the query. None of the 3 overlaps the 10 benchmark queries (asserted below).


In [16]:
import json

FEWSHOT_IDX = [1, 10, 47]

with open("instruction_dataset.jsonl") as f:
    all_pairs = [json.loads(l) for l in f]

fewshot_examples = [all_pairs[i] for i in FEWSHOT_IDX]
benchmark_source_idx = {b["source_idx"] for b in benchmark}
assert not (set(FEWSHOT_IDX) & benchmark_source_idx), "Few-shot examples must not overlap the benchmark set"

print("Few-shot examples used:")
for i, ex in zip(FEWSHOT_IDX, fewshot_examples):
    print(f"\n[{i}] Q: {ex['instruction']}\n    A: {ex['response']}")


Few-shot examples used:

[1] Q: Describe how BERT chooses and corrupts tokens for its masked language model objective.
    A: BERT selects 15 percent of the WordPiece positions at random for prediction. Of those positions, 80 percent are replaced by the [MASK] token, 10 percent by a random token and 10 percent are left unchanged, which reduces the mismatch with fine-tuning where [MASK] never appears.

[10] Q: What was the main goal of the LLaMA model family?
    A: LLaMA set out to show that state-of-the-art language models can be trained using only publicly available data. The family spans 7 to 65 billion parameters, and the 13 billion parameter model outperforms the much larger GPT-3 on most benchmarks while being small enough to run on a single GPU.

[47] Q: How does FlashAttention handle the backward pass efficiently?
    A: Instead of storing the large attention matrix for the backward pass, FlashAttention keeps only the output and the softmax normalisation statistics. It then rec

In [17]:
def build_fewshot_prompt(query):
    blocks = SYSTEM_PROMPT + "\n\n"
    blocks += "".join(
        f"### Question:\n{ex['instruction']}\n\n### Answer:\n{ex['response']}\n\n"
        for ex in fewshot_examples
    )
    blocks += f"### Question:\n{query}\n\n### Answer:\n"
    return blocks

def pattern_few_shot(query):
    full_prompt = build_fewshot_prompt(query)
    text, p_tok, g_tok, lat = call_generator_raw(full_prompt, stop_strings=["### Question:"])
    return text, p_tok + g_tok, lat

print(build_fewshot_prompt("<example query goes here>"))


You are a research assistant that answers questions about deep learning and large language model papers accurately and concisely.

### Question:
Describe how BERT chooses and corrupts tokens for its masked language model objective.

### Answer:
BERT selects 15 percent of the WordPiece positions at random for prediction. Of those positions, 80 percent are replaced by the [MASK] token, 10 percent by a random token and 10 percent are left unchanged, which reduces the mismatch with fine-tuning where [MASK] never appears.

### Question:
What was the main goal of the LLaMA model family?

### Answer:
LLaMA set out to show that state-of-the-art language models can be trained using only publicly available data. The family spans 7 to 65 billion parameters, and the 13 billion parameter model outperforms the much larger GPT-3 on most benchmarks while being small enough to run on a single GPU.

### Question:
How does FlashAttention handle the backward pass efficiently?

### Answer:
Instead of stori

### A1. Instruction template

A role + numbered-steps template suited to scientific-paper questions (approved wording).


In [18]:
INSTRUCTION_TEMPLATE = (
    "You are a research assistant answering questions about scientific papers on "
    "language models. Follow these steps:\n"
    "1. Identify the key technical concept the question is asking about.\n"
    "2. Recall the relevant fact, method, or result from the paper.\n"
    "3. State the answer in 1-3 concise sentences, using precise technical terms.\n\n"
    "Question: {query}\n"
    "Answer:"
)

print(INSTRUCTION_TEMPLATE)

def pattern_instruction(query):
    full_prompt = INSTRUCTION_TEMPLATE.format(query=query)
    text, p_tok, g_tok, lat = call_generator_raw(full_prompt, stop_strings=["### Question:", "\nQuestion:"])
    return text, p_tok + g_tok, lat


You are a research assistant answering questions about scientific papers on language models. Follow these steps:
1. Identify the key technical concept the question is asking about.
2. Recall the relevant fact, method, or result from the paper.
3. State the answer in 1-3 concise sentences, using precise technical terms.

Question: {query}
Answer:


### A2. Self-critique

Three sequential generator calls: answer -> critique of that answer -> revised answer using the critique. Each output feeds the next prompt. `tokens_used` and `latency_sec` sum across all three calls.


In [19]:
CRITIQUE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Answer: {answer}\n\n"
    "Critique this answer for factual accuracy and completeness in 1-2 sentences. "
    "Point out anything incorrect or missing.\n"
    "Critique:"
)

REVISE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Original answer: {answer}\n"
    "Critique: {critique}\n\n"
    "Using the critique, write an improved, concise answer (1-3 sentences).\n"
    "Revised answer:"
)

self_critique_trace = []

def _self_critique_full(query):
    answer1, p1, g1, l1 = call_generator(query)
    critique_prompt = CRITIQUE_PROMPT_TEMPLATE.format(query=query, answer=answer1)
    critique, p2, g2, l2 = call_generator_raw(critique_prompt, stop_strings=["### Question:", "\nQuestion:"])
    revise_prompt = REVISE_PROMPT_TEMPLATE.format(query=query, answer=answer1, critique=critique)
    revised, p3, g3, l3 = call_generator_raw(revise_prompt, stop_strings=["### Question:", "\nQuestion:"])
    tokens_used = (p1 + g1) + (p2 + g2) + (p3 + g3)
    latency = l1 + l2 + l3
    self_critique_trace.append({
        "query": query, "first_answer": answer1, "critique": critique, "revised_answer": revised,
    })
    return revised, tokens_used, latency

def pattern_self_critique(query):
    return _self_critique_full(query)


### A3. Structured output

JSON schema for this domain (approved): `{"source_paper", "claim", "evidence"}` — `source_paper` is the best-guess paper/technique this relates to (or `"unknown"`), `claim` is the specific factual claim being asked about, `evidence` is a 1-3 sentence explanation supporting it. The model is prompted to return only JSON matching this schema; parsing is attempted on the raw text, then (documented fallback) on the substring between the first `{` and the last `}` if the raw text failed to parse; on failure the model is re-prompted once with the parse error included.


In [20]:
STRUCTURED_SCHEMA_FIELDS = ["source_paper", "claim", "evidence"]

STRUCTURED_PROMPT_TEMPLATE = (
    "Answer the question below. Respond with ONLY a JSON object with exactly these three keys, "
    "spelled exactly as shown below, each key in double quotes: "
    '"source_paper" (the paper or technique this relates to, or "unknown" if unclear), '
    '"claim" (the specific factual claim the question is about), '
    '"evidence" (a 1-3 sentence explanation supporting the claim). '
    "Use source_paper with an underscore, not sourcePaper. Use double quotes around every key and "
    "string value, not single quotes. Do not add any other keys. "
    "Do not include any text outside the JSON object.\n\n"
    "Example:\n"
    "Question: What does the attention mechanism in a Transformer compute?\n"
    'JSON: {{"source_paper": "Transformer", "claim": "Attention computes a weighted sum of value '
    'vectors using similarity between query and key vectors.", "evidence": "This lets each token '
    'attend to the most relevant other tokens regardless of their distance in the sequence."}}\n\n'
    "Question: {query}\n"
    "JSON:"
)

print(STRUCTURED_PROMPT_TEMPLATE)

def extract_json_object(text):
    """Documented fallback: if extra text surrounds the JSON object, extract the substring
    between the first '{' and the last '}' before parsing again."""
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return text
    return text[start:end + 1]

def try_parse_structured(raw_text):
    last_error = "no '{' / '}' found"
    for candidate in (raw_text, extract_json_object(raw_text)):
        try:
            obj = json.loads(candidate)
        except json.JSONDecodeError as e:
            last_error = str(e)
            continue
        missing = [f for f in STRUCTURED_SCHEMA_FIELDS if f not in obj]
        if missing:
            last_error = f"missing required fields: {missing}"
            continue
        return obj, None
    return None, last_error


Answer the question below. Respond with ONLY a JSON object with exactly these three keys, spelled exactly as shown below, each key in double quotes: "source_paper" (the paper or technique this relates to, or "unknown" if unclear), "claim" (the specific factual claim the question is about), "evidence" (a 1-3 sentence explanation supporting the claim). Use source_paper with an underscore, not sourcePaper. Use double quotes around every key and string value, not single quotes. Do not add any other keys. Do not include any text outside the JSON object.

Example:
Question: What does the attention mechanism in a Transformer compute?
JSON: {{"source_paper": "Transformer", "claim": "Attention computes a weighted sum of value vectors using similarity between query and key vectors.", "evidence": "This lets each token attend to the most relevant other tokens regardless of their distance in the sequence."}}

Question: {query}
JSON:


In [21]:
structured_trace = []

def pattern_structured(query):
    prompt = STRUCTURED_PROMPT_TEMPLATE.format(query=query)
    raw, p1, g1, l1 = call_generator_raw(prompt, stop_strings=["\nQuestion:"])
    obj, error = try_parse_structured(raw)
    tokens_used = p1 + g1
    latency = l1
    trace = {"query": query, "attempt_1_raw": raw, "attempt_1_error": error, "attempt_1_parsed": obj}

    if obj is None:
        retry_prompt = (
            "Your previous response could not be parsed as JSON matching the schema.\n"
            f"Previous response: {raw}\n"
            f"Parse error: {error}\n"
            "Return ONLY a valid JSON object with exactly these three keys, spelled exactly as "
            'shown, each in double quotes: "source_paper", "claim", "evidence". Use an underscore '
            "in source_paper, not camelCase sourcePaper. Use double quotes, not single quotes. "
            "No text outside the JSON object.\n\n"
            'Example: {"source_paper": "Transformer", "claim": "...", "evidence": "..."}\n\n'
            f"Question: {query}\n"
            "JSON:"
        )
        raw2, p2, g2, l2 = call_generator_raw(retry_prompt, stop_strings=["\nQuestion:"])
        obj2, error2 = try_parse_structured(raw2)
        tokens_used += p2 + g2
        latency += l2
        trace.update({"attempt_2_raw": raw2, "attempt_2_error": error2, "attempt_2_parsed": obj2})
        obj = obj2

    structured_trace.append(trace)
    answer_text = json.dumps(obj) if obj is not None else raw
    return answer_text, tokens_used, latency


### A4. LLM judge

Judge model: **`Qwen/Qwen2.5-0.5B-Instruct`** — a local dev-scale placeholder standing in for the brief's `Qwen2.5-7B-Instruct` (that model needs roughly 28 GB RAM in fp32 on CPU; this machine has 16 GB and no CUDA GPU, so it cannot load here). The placeholder validates the judge pipeline (prompt format, `SCORE`/`REASON` parsing, retry-on-failure); final scores for submission are produced by re-running this same code with `JUDGE_MODEL_ID` pointed at `Qwen/Qwen2.5-7B-Instruct` on Prayogshala. Judge decoding is deterministic (`do_sample=False`). Judge prompt/generated tokens are **not** counted in any pattern's `tokens_used`.


In [22]:
from transformers import AutoModelForCausalLM as _AutoModelForCausalLM
import re

JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = _AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_ID, torch_dtype=torch.bfloat16)
judge_model.to(DEVICE)
judge_model.eval()
print(f"Judge model loaded: {JUDGE_MODEL_ID}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Judge model loaded: Qwen/Qwen2.5-7B-Instruct


In [23]:
JUDGE_SYSTEM_PROMPT = (
    "You are a strict grading assistant for question-answering about scientific papers on language models."
)

JUDGE_USER_TEMPLATE = (
    "Question: {query}\n"
    "Known correct answer: {known_answer}\n"
    "Candidate answer: {candidate_answer}\n\n"
    "Score the candidate answer using this scale:\n"
    "2 = correct (all key facts present, nothing wrong)\n"
    "1 = partly correct (some key facts missing, nothing wrong)\n"
    "0 = wrong (contains incorrect information)\n\n"
    "Respond in exactly this format, each on its own line:\n"
    "SCORE: <0, 1, or 2>\n"
    "REASON: <one sentence>"
)

print(JUDGE_SYSTEM_PROMPT)
print()
print(JUDGE_USER_TEMPLATE)

SCORE_RE = re.compile(r"SCORE:\s*([012])", re.IGNORECASE)
REASON_RE = re.compile(r"REASON:\s*(.+)", re.IGNORECASE)

def judge_call(query, known_answer, candidate_answer):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": JUDGE_USER_TEMPLATE.format(
            query=query, known_answer=known_answer, candidate_answer=candidate_answer
        )},
    ]
    prompt_text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = judge_tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out_ids = judge_model.generate(
            **inputs, max_new_tokens=60, do_sample=False, pad_token_id=judge_tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - t0
    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    raw = judge_tokenizer.decode(gen_ids, skip_special_tokens=True)
    return raw, latency, inputs["input_ids"].shape[1], gen_ids.shape[0]

def parse_judge_output(raw):
    score_m = SCORE_RE.search(raw)
    reason_m = REASON_RE.search(raw)
    if score_m and reason_m:
        return int(score_m.group(1)), reason_m.group(1).strip(), True
    return None, raw.strip(), False

def judge(query, known_answer, candidate_answer):
    raw, lat1, pt1, gt1 = judge_call(query, known_answer, candidate_answer)
    score, reason, ok = parse_judge_output(raw)
    attempts = [{"raw": raw, "latency": lat1, "prompt_tokens": pt1, "gen_tokens": gt1}]
    if not ok:
        raw2, lat2, pt2, gt2 = judge_call(query, known_answer, candidate_answer)
        score, reason, ok2 = parse_judge_output(raw2)
        attempts.append({"raw": raw2, "latency": lat2, "prompt_tokens": pt2, "gen_tokens": gt2})
        if not ok2:
            score, reason = None, f"UNPARSEABLE after retry: {raw2!r}"
    judge_tokens = sum(a["prompt_tokens"] + a["gen_tokens"] for a in attempts)
    judge_latency = sum(a["latency"] for a in attempts)
    return score, reason, len(attempts), judge_tokens, judge_latency


You are a strict grading assistant for question-answering about scientific papers on language models.

Question: {query}
Known correct answer: {known_answer}
Candidate answer: {candidate_answer}

Score the candidate answer using this scale:
2 = correct (all key facts present, nothing wrong)
1 = partly correct (some key facts missing, nothing wrong)
0 = wrong (contains incorrect information)

Respond in exactly this format, each on its own line:
SCORE: <0, 1, or 2>
REASON: <one sentence>


Run all five patterns on the 10 benchmark queries, judge every answer, and save the raw per-query results to `results_partA.csv`.


In [24]:
import pandas as pd

PATTERNS_A = {
    "zero_shot": pattern_zero_shot,
    "few_shot": pattern_few_shot,
    "instruction": pattern_instruction,
    "self_critique": pattern_self_critique,
    "structured": pattern_structured,
}

records = []
for q in benchmark:
    query, known = q["instruction"], q["response"]
    for pname, pfunc in PATTERNS_A.items():
        answer, tokens_used, latency = pfunc(query)
        score, reason, judge_attempts, judge_tokens, judge_latency = judge(query, known, answer)
        records.append({
            "source_idx": q["source_idx"], "type": q["type"], "pattern": pname,
            "query": query, "known_answer": known, "pattern_answer": answer,
            "tokens_used": tokens_used, "latency_sec": latency,
            "judge_score": score, "judge_reason": reason, "judge_attempts": judge_attempts,
            "judge_tokens": judge_tokens, "judge_latency_sec": judge_latency,
        })
        print(f"[{pname:13s}] idx={q['source_idx']:>2} score={score} tokens={tokens_used:>4} latency={latency:6.1f}s")

results_partA = pd.DataFrame(records)
results_partA["judge_score"] = pd.to_numeric(results_partA["judge_score"], errors="coerce")
results_partA.to_csv("results_partA.csv", index=False)
print(f"\nSaved {len(results_partA)} rows to results_partA.csv")


/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[zero_shot    ] idx=27 score=0 tokens=  77 latency=   0.9s
[few_shot     ] idx=27 score=0 tokens= 367 latency=   1.4s
[instruction  ] idx=27 score=0 tokens= 156 latency=   2.0s
[self_critique] idx=27 score=0 tokens= 749 latency=  12.3s
[structured   ] idx=27 score=0 tokens= 296 latency=   5.7s
[zero_shot    ] idx=30 score=0 tokens= 182 latency=   3.9s
[few_shot     ] idx=30 score=0 tokens= 403 latency=   2.6s
[instruction  ] idx=30 score=1 tokens= 154 latency=   2.1s
[self_critique] idx=30 score=0 tokens=1030 latency=  15.6s
[structured   ] idx=30 score=0 tokens= 640 latency=  11.7s
[zero_shot    ] idx=25 score=0 tokens= 116 latency=   2.2s
[few_shot     ] idx=25 score=0 tokens= 392 latency=   5.2s
[instruction  ] idx=25 score=0 tokens= 188 latency=   5.8s
[self_critique] idx=25 score=0 tokens= 661 latency=  13.7s
[structured   ] idx=25 score=0 tokens= 346 latency=   6.0s
[zero_shot    ] idx=57 score=0 tokens=  95 latency=   1.3s
[few_shot     ] idx=57 score=0 tokens= 384 latency=   1.

### Part A — results table


In [25]:
partA_table = results_partA.groupby("pattern").agg(
    avg_score=("judge_score", "mean"),
    avg_tokens=("tokens_used", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
).reindex(["zero_shot", "few_shot", "instruction", "self_critique", "structured"])
partA_table


,avg_score,avg_tokens,avg_latency_sec
pattern,,,
zero_shot,0.1,109.6,1.838714
few_shot,0.0,393.7,3.517927
instruction,0.2,157.6,3.406713
self_critique,0.1,697.1,12.858035
structured,0.0,479.1,8.780914


### Part A — spot-check

5 randomly selected judged answers (fixed seed) for manual review.


In [26]:
spot_check_sample = results_partA.sample(n=5, random_state=123).reset_index(drop=True)
spot_check_sample[["source_idx", "pattern", "query", "known_answer", "pattern_answer", "judge_score", "judge_reason"]]


,source_idx,pattern,query,known_answer,pattern_answer,judge_score,judge_reason
0,25,zero_shot,What are the three main innovations introduced...,QLoRA introduces the 4-bit NormalFloat data ty...,QLoRA is a new deep learning framework that us...,0,The candidate answer contains incorrect inform...
1,25,self_critique,What are the three main innovations introduced...,QLoRA introduces the 4-bit NormalFloat data ty...,QLoRA is a new deep learning framework that us...,0,The candidate answer contains incorrect inform...
2,14,zero_shot,How does the LLaMA tokenizer handle numbers an...,LLaMA tokenizes text with byte-pair encoding i...,The LLaMA tokenizer uses a fixed-length tokeni...,0,The candidate answer contains incorrect inform...
3,53,few_shot,How does DPO compare with PPO-based RLHF in th...,"Across sentiment control, summarisation and si...",DPO is 10 times faster and 100 times more accu...,0,The candidate answer contains incorrect inform...
4,57,self_critique,How does PPO-ptx differ from the plain PPO mod...,Plain PPO training caused performance regressi...,The PPO-ptx model uses a different loss functi...,0,The answer is largely incorrect and lacks cont...


Vijay's verdict: agree with the judge on all 5 rows above -- each is a genuinely wrong or fabricated answer (QLoRA's innovations invented twice over, including a non-existent "QLoQA" framework; LLaMA's tokenizer behaviour reversed; invented DPO-vs-PPO performance numbers; PPO-ptx's actual mechanism -- mixing pre-training gradients into PPO updates -- never mentioned), and the judge's stated reason correctly identifies what's wrong in every case.

| row | agree / disagree |
|---|---|
| 0 | agree |
| 1 | agree |
| 2 | agree |
| 3 | agree |
| 4 | agree |

Total agree: 5 / 5


### Part A — self-critique traces

First answer, critique, and revised answer for 2 benchmark queries.


In [27]:
for trace in self_critique_trace[:2]:
    print("=" * 80)
    print("Query:   ", trace["query"])
    print("\nFirst answer:\n", trace["first_answer"])
    print("\nCritique:\n", trace["critique"])
    print("\nRevised answer:\n", trace["revised_answer"])
    print()


Query:    How much memory does double quantization save in QLoRA?

First answer:
 The double quantization of the QLoRA model saves 1.5 GB of memory, which is equivalent to 100 GB of RAM.

Critique:
 The double quantization of the QLoRA model saves 1.5 GB of memory, which is equivalent to 100 GB of RAM. This is a very large amount of memory, and it would be difficult to store all of the data in a single file. However, the double quantization of the model also reduces the number of parameters by 100, which means that the model can fit more data without needing to store more parameters. This is a trade-off between memory and performance, and the trade-off is not always the same for every model.

Conclusion: The double quantization of the QLoRA model saves 1.5 GB of memory, which is equivalent to 100 GB of RAM. This is a large amount of memory, and it would be difficult to store all of the data in a single file. However, the double quantization of the model also reduces the number of param

In [28]:
import re as _re

def _word_overlap(a, b):
    wa = set(_re.findall(r"[a-zA-Z0-9]+", a.lower()))
    wb = set(_re.findall(r"[a-zA-Z0-9]+", b.lower()))
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

for trace in self_critique_trace[:2]:
    known = next(b["response"] for b in benchmark if b["instruction"] == trace["query"])
    overlap_first = _word_overlap(trace["first_answer"], known)
    overlap_revised = _word_overlap(trace["revised_answer"], known)
    # A small epsilon avoids ever printing a confusing "rose from 0.15 to 0.15" when the two
    # values are nearly but not exactly equal and round to the same 2 decimal places.
    if overlap_revised > overlap_first + 0.005:
        verdict = (f"word overlap with the known answer rose from {overlap_first:.3f} to "
                   f"{overlap_revised:.3f}, suggesting the revision improved the answer")
    elif overlap_revised < overlap_first - 0.005:
        verdict = (f"word overlap with the known answer fell from {overlap_first:.3f} to "
                   f"{overlap_revised:.3f}, suggesting the revision did not improve the answer")
    else:
        verdict = (f"word overlap with the known answer stayed at about {overlap_first:.3f}, "
                   f"showing no clear improvement by this measure")
    print(f"DRAFT -- review before submission: for {trace['query'][:70]!r}..., {verdict}.")


DRAFT -- review before submission: for 'How much memory does double quantization save in QLoRA?'..., word overlap with the known answer stayed at about 0.154, showing no clear improvement by this measure.
DRAFT -- review before submission: for 'What is prefix-tuning and how many parameters does it train?'..., word overlap with the known answer stayed at about 0.141, showing no clear improvement by this measure.


### Part A — structured-output examples

3 valid outputs, and (if one occurred naturally across the 10 queries) one real parse failure with its retry.


In [29]:
valid = [t for t in structured_trace if t.get("attempt_1_parsed") is not None]
failed_first = [t for t in structured_trace if t.get("attempt_1_parsed") is None]

print(f"Parsed on first attempt: {len(valid)}/{len(structured_trace)}")
print(f"Failed on first attempt (retried): {len(failed_first)}/{len(structured_trace)}")

print("\n--- 3 valid outputs ---")
for t in valid[:3]:
    print("\nQuery:", t["query"])
    print("Parsed:", t["attempt_1_parsed"])

if failed_first:
    t = failed_first[0]
    print("\n--- Real parse failure (attempt 1) and its retry ---")
    print("Query:", t["query"])
    print("Raw (attempt 1):", t["attempt_1_raw"])
    print("Parse error:", t["attempt_1_error"])
    print("Raw (attempt 2, after retry prompt with the error):", t.get("attempt_2_raw"))
    print("Parsed (attempt 2):", t.get("attempt_2_parsed"))
else:
    print("\nNo natural parse failure occurred across these 10 queries -- per the brief, "
          "not attempting to provoke one without asking first.")


Parsed on first attempt: 5/10
Failed on first attempt (retried): 5/10

--- 3 valid outputs ---

Query: How much memory does double quantization save in QLoRA?
Parsed: {'source_paper': 'Transformer', 'claim': 'Double quantization saves 100x more memory than single quantization, and 10x more than single-bit quantization.', 'evidence': 'The memory savings are 100x because each bit is replaced by 2 bits, and 10x because each bit is replaced by 4 bits.'}

Query: What are the three main innovations introduced by QLoRA?
Parsed: {'source_paper': 'QLoRA', 'claim': 'The first is a new attention mechanism that computes a weighted sum of value vectors using similarity between query and key vectors.', 'evidence': 'The second is a new attention model that computes a weighted sum of value vectors using the same query and key vectors, but with a different attention model for each query and key. This allows the attention model to be trained on a different set of queries and keys, and to be shared betwe

### Part A — draft verdict: does self-critique justify its extra tokens over zero-shot?


In [30]:
sc_avg_score = partA_table.loc["self_critique", "avg_score"]
zs_avg_score = partA_table.loc["zero_shot", "avg_score"]
sc_tokens = partA_table.loc["self_critique", "avg_tokens"]
zs_tokens = partA_table.loc["zero_shot", "avg_tokens"]

if sc_avg_score > zs_avg_score:
    comparison = f"self-critique improved the average judge score from {zs_avg_score:.2f} to {sc_avg_score:.2f}"
    judgement = "justifies" if (sc_avg_score - zs_avg_score) >= 0.3 else "only marginally justifies"
elif sc_avg_score == zs_avg_score:
    comparison = f"self-critique matched zero-shot's average judge score of {zs_avg_score:.2f}"
    judgement = "does not justify"
else:
    comparison = f"self-critique scored lower on average ({sc_avg_score:.2f} vs {zs_avg_score:.2f})"
    judgement = "does not justify"

draft_text = (
    "DRAFT -- review before submission: "
    f"{comparison}, using {sc_tokens:.0f} tokens on average versus {zs_tokens:.0f} for zero-shot "
    f"({sc_tokens / zs_tokens:.1f}x). This {judgement} the extra token cost."
)
print(draft_text)


DRAFT -- review before submission: self-critique matched zero-shot's average judge score of 0.10, using 697 tokens on average versus 110 for zero-shot (6.4x). This does not justify the extra token cost.


## Part B — Reasoning patterns and a chain

Two multi-step problems from this project's own domain, each with a known answer grounded in the source papers and in this project's own trained adapter (not just computed by hand -- the LoRA parameter count below was cross-checked against a real `peft.get_peft_model(...).print_trainable_parameters()` call on the actual `cpt_final_model`). Used throughout B1 and B3.


In [31]:
PROBLEMS_B = [
    {
        "id": 1,
        "question": (
            "SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads "
            "(grouped-query attention), and 32 transformer layers. A LoRA adapter with "
            "rank r=16 is applied to the q_proj and v_proj linear layers in every layer. "
            "How many trainable parameters does LoRA add in total?"
        ),
        "known_answer": (
            "1,638,400. head_dim = 960 / 15 = 64. q_proj output dim = 15 * 64 = 960 (full "
            "attention dimension), v_proj output dim = 5 * 64 = 320 (reduced by GQA). LoRA "
            "params per module = r * (in_features + out_features): q_proj = 16 * (960 + 960) "
            "= 30,720; v_proj = 16 * (960 + 320) = 20,480. Per layer = 51,200. Across 32 "
            "layers = 1,638,400."
        ),
    },
    {
        "id": 2,
        "question": (
            "Chinchilla's own compute-optimal training used 1.4 trillion tokens for its "
            "70-billion-parameter model. GPT-3 was trained on 300 billion tokens for its "
            "175-billion-parameter model. Compute each model's tokens-per-parameter ratio, "
            "and state by what factor GPT-3's training ratio falls short of Chinchilla's "
            "compute-optimal ratio."
        ),
        "known_answer": (
            "Chinchilla: 1.4e12 / 70e9 = 20 tokens/parameter. GPT-3: 300e9 / 175e9 ~= 1.71 "
            "tokens/parameter. Factor short: 20 / 1.71 ~= 11.7x -- GPT-3 was trained on "
            "roughly 11.7x fewer tokens per parameter than Chinchilla's compute-optimal ratio."
        ),
    },
]

for p in PROBLEMS_B:
    print(f"Problem {p['id']}: {p['question']}\nKnown answer: {p['known_answer']}\n")


Problem 1: SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads (grouped-query attention), and 32 transformer layers. A LoRA adapter with rank r=16 is applied to the q_proj and v_proj linear layers in every layer. How many trainable parameters does LoRA add in total?
Known answer: 1,638,400. head_dim = 960 / 15 = 64. q_proj output dim = 15 * 64 = 960 (full attention dimension), v_proj output dim = 5 * 64 = 320 (reduced by GQA). LoRA params per module = r * (in_features + out_features): q_proj = 16 * (960 + 960) = 30,720; v_proj = 16 * (960 + 320) = 20,480. Per layer = 51,200. Across 32 layers = 1,638,400.

Problem 2: Chinchilla's own compute-optimal training used 1.4 trillion tokens for its 70-billion-parameter model. GPT-3 was trained on 300 billion tokens for its 175-billion-parameter model. Compute each model's tokens-per-parameter ratio, and state by what factor GPT-3's training ratio falls short of Chinchilla's compute-optimal ratio.
Known answer: Chinchilla: 1.

### B1. Chain-of-thought and self-consistency

Chain-of-thought asks the model to reason step by step and end with a `Final answer:` line, which is the only part judged (per the brief). Self-consistency samples 5 chain-of-thought completions per problem at temperature 0.7 (`do_sample=True`), extracts each sample's final answer, and takes the majority vote -- ties broken by the earliest sample in the largest group. The zero-shot pattern (A0) is reused unchanged as the baseline row.

An earlier run used `max_new_tokens=300` for these multi-step calls after seeing outputs cut off mid-calculation at 200 tokens. Reverted to the brief's fixed `max_new_tokens=200` (used identically everywhere, including Part A) for strict compliance with the Setup section's decoding-settings instruction; `repetition_penalty`/`no_repeat_ngram_size` (see Step 2) are the fix now applied for the actual root cause -- the model wasting its budget on repeated phrases -- rather than compensating for it with a larger budget.


In [32]:
FINAL_ANSWER_RE = re.compile(r"Final answer:\s*(.+)", re.IGNORECASE)

COT_PROMPT_TEMPLATE = (
    "Question: {query}\n\n"
    "Reason step by step, showing each step of your calculation or reasoning. "
    "End your response with a final line in exactly this format:\n"
    "Final answer: <your answer>\n\n"
    "Reasoning:"
)

print(COT_PROMPT_TEMPLATE)

def extract_final_answer(text):
    m = FINAL_ANSWER_RE.search(text)
    return m.group(1).strip() if m else text.strip()

cot_trace = []

def pattern_cot(query):
    prompt = COT_PROMPT_TEMPLATE.format(query=query)
    text, p_tok, g_tok, lat = call_generator_raw(prompt, stop_strings=["\nQuestion:"])
    cot_trace.append({"query": query, "reasoning": text, "final_answer": extract_final_answer(text)})
    return text, p_tok + g_tok, lat


Question: {query}

Reason step by step, showing each step of your calculation or reasoning. End your response with a final line in exactly this format:
Final answer: <your answer>

Reasoning:


In [33]:
self_consistency_trace = []

def _self_consistency_full(query, n_samples=5, temperature=0.7):
    prompt = COT_PROMPT_TEMPLATE.format(query=query)
    samples = []
    total_tokens = 0
    total_latency = 0.0
    for _ in range(n_samples):
        text, p_tok, g_tok, lat = call_generator_raw(
            prompt, stop_strings=["\nQuestion:"], do_sample=True, temperature=temperature,
        )
        samples.append(extract_final_answer(text))
        total_tokens += p_tok + g_tok
        total_latency += lat

    counts = Counter(samples)
    max_count = max(counts.values())
    majority_answer = next(s for s in samples if counts[s] == max_count)  # earliest sample from largest group

    self_consistency_trace.append({
        "query": query, "samples": samples, "counts": dict(counts), "majority_answer": majority_answer,
    })
    return majority_answer, total_tokens, total_latency

def pattern_self_consistency(query):
    return _self_consistency_full(query)


In [34]:
def build_b1_table(records, order):
    rows = {}
    for pname in order:
        sub = [r for r in records if r["pattern"] == pname]
        if not sub:
            continue
        p1 = next((r["judge_score"] for r in sub if r["problem_id"] == 1), None)
        p2 = next((r["judge_score"] for r in sub if r["problem_id"] == 2), None)
        avg_tok = sum(r["tokens_used"] for r in sub) / len(sub)
        rows[pname] = {"problem_1_score": p1, "problem_2_score": p2, "avg_tokens": avg_tok}
    return pd.DataFrame(rows).T

b1_records = []
B1_PATTERNS = {
    "zero_shot": pattern_zero_shot,
    "chain_of_thought": pattern_cot,
    "self_consistency": pattern_self_consistency,
}

for p in PROBLEMS_B:
    query, known = p["question"], p["known_answer"]
    for pname, pfunc in B1_PATTERNS.items():
        answer, tokens_used, latency = pfunc(query)
        judged_answer = extract_final_answer(answer) if pname == "chain_of_thought" else answer
        score, reason, judge_attempts, judge_tokens, judge_latency = judge(query, known, judged_answer)
        b1_records.append({
            "problem_id": p["id"], "pattern": pname, "query": query,
            "answer": judged_answer, "tokens_used": tokens_used, "latency_sec": latency,
            "judge_score": score, "judge_reason": reason,
        })
        print(f"[{pname:17s}] problem={p['id']} score={score} tokens={tokens_used:>4}")

b1_table = build_b1_table(b1_records, ["zero_shot", "chain_of_thought", "self_consistency"])
b1_table


/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[zero_shot        ] problem=1 score=0 tokens= 286
[chain_of_thought ] problem=1 score=0 tokens= 325
[self_consistency ] problem=1 score=0 tokens=1519
[zero_shot        ] problem=2 score=0 tokens= 283
[chain_of_thought ] problem=2 score=0 tokens= 329
[self_consistency ] problem=2 score=0 tokens=1624


,problem_1_score,problem_2_score,avg_tokens
zero_shot,0.0,0.0,284.5
chain_of_thought,0.0,0.0,327.0
self_consistency,0.0,0.0,1571.5


### B1 — self-consistency votes

The 5 sampled final answers and the vote for each problem.


In [35]:
for trace in self_consistency_trace:
    print(f"Query: {trace['query'][:80]}...")
    for i, s in enumerate(trace["samples"], start=1):
        print(f"  sample {i}: {s}")
    print(f"  vote counts: {trace['counts']}")
    print(f"  majority answer: {trace['majority_answer']}\n")


Query: SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads (grouped...
  sample 1: This is a classic example of a single-parameter linear layer in a LoRA layer being used in a multi-head Attention computation. To add the LoRA adapter, we add one new layer whose role is to apply the LoRA to each query and value head. We then multiply the size of each new head by 20, which gives 16. That’s where the “50000” comes from.
  sample 2: 2
  sample 3: For each hidden size, you need to calculate the new attention heads and compute the new query and value attention layers for each head, which requires 32-bit arithmetic and 256 attention heads. You also need to add the 32-bit attention heads to the query and value layers in every layer, which involves 32-bit arithmetic. There are 10-bit attention heads for each 64-bit query and value, which means you need to pack 360 attention heads per layer and calculate 960 attention heads per layer, so you need to perform 360*960*640=324,00

### B2. ReAct

A local AST-based calculator tool (only `+ - * / **`, no `eval()`), and a Thought/Action/Observation loop that stops when the model gives a final answer or after 5 cycles. Run on Problem 2 (the Chinchilla-vs-GPT-3 ratio), which needs 3 separate divisions -- a natural fit for a tool-use loop.


In [36]:
import ast
import operator

_ALLOWED_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

def calculator(expression: str) -> str:
    """Safely evaluates a numeric expression (+, -, *, /, **) via an AST whitelist -- no eval()."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception as e:
        return f"ERROR: {e}"

print(calculator("1.4e12 / 70e9"))
print(calculator("300e9 / 175e9"))
print(calculator("import os"))


20.0
1.7142857142857142
ERROR: invalid syntax (<unknown>, line 1)


In [37]:
REACT_SYSTEM_PROMPT = (
    "Answer the question by reasoning step by step and using the calculator tool for any "
    "arithmetic. This kind of question usually needs several separate calculator steps before "
    "you have enough information for a final answer -- do not give a final answer until you "
    "have used the calculator for every number you need. Each turn, respond with ONLY one of "
    "the following:\n"
    "Thought: <your reasoning about what to do next>\n"
    "Action: calculator(<a numeric expression>)\n"
    "or, once you know the final answer:\n"
    "Thought: <your reasoning>\n"
    "Final answer: <your answer>\n\n"
    "Example:\n"
    "Question: Model A has 10 layers of 200 parameters each, and Model B has 5 layers of 300 "
    "parameters each. What is the ratio of Model A's total parameters to Model B's total "
    "parameters?\n"
    "Thought: First I need Model A's total parameters.\n"
    "Action: calculator(10 * 200)\n"
    "Observation: 2000\n"
    "Thought: Now I need Model B's total parameters.\n"
    "Action: calculator(5 * 300)\n"
    "Observation: 1500\n"
    "Thought: Now I can compute the ratio.\n"
    "Action: calculator(2000 / 1500)\n"
    "Observation: 1.3333333333333333\n"
    "Thought: I now have the ratio.\n"
    "Final answer: 1.33\n"
)

ACTION_RE = re.compile(r"Action:\s*calculator\((.+?)\)", re.IGNORECASE)
REACT_FINAL_RE = re.compile(r"Final answer:\s*(.+)", re.IGNORECASE)

react_trace = []

def pattern_react(query, max_cycles=5):
    transcript = f"{REACT_SYSTEM_PROMPT}\nQuestion: {query}\n"
    cycles = []
    total_tokens, total_latency = 0, 0.0
    final_answer = None

    for _ in range(max_cycles):
        text, p_tok, g_tok, lat = call_generator_raw(
            transcript, stop_strings=["\nObservation:", "\nQuestion:"],
        )
        total_tokens += p_tok + g_tok
        total_latency += lat

        final_m = REACT_FINAL_RE.search(text)
        action_m = ACTION_RE.search(text)
        if final_m:
            final_answer = final_m.group(1).strip()
            cycles.append({"thought_action": text.strip(), "observation": None})
            transcript += text.strip() + "\n"
            break
        elif action_m:
            observation = calculator(action_m.group(1).strip())
            cycles.append({"thought_action": text.strip(), "observation": observation})
            transcript += text.strip() + f"\nObservation: {observation}\n"
        else:
            cycles.append({"thought_action": text.strip(), "observation": None})
            transcript += text.strip() + "\n"
            break

    react_trace.append({"query": query, "cycles": cycles, "final_answer": final_answer})
    answer = final_answer if final_answer is not None else (cycles[-1]["thought_action"] if cycles else "")
    return answer, total_tokens, total_latency


In [38]:
react_query = PROBLEMS_B[1]["question"]
react_answer, react_tokens, react_latency = pattern_react(react_query)

print(f"Query: {react_query}\n")
for i, c in enumerate(react_trace[-1]["cycles"], start=1):
    print(f"--- Cycle {i} ---")
    print(c["thought_action"])
    if c["observation"] is not None:
        print(f"Observation: {c['observation']}")
    print()

print(f"Final answer: {react_answer}")
print(f"tokens_used={react_tokens} latency_sec={react_latency:.2f} cycles={len(react_trace[-1]['cycles'])}")

react_score, react_reason, _, _, _ = judge(react_query, PROBLEMS_B[1]["known_answer"], react_answer)
print(f"\njudge_score={react_score} ({react_reason})")


Query: Chinchilla's own compute-optimal training used 1.4 trillion tokens for its 70-billion-parameter model. GPT-3 was trained on 300 billion tokens for its 175-billion-parameter model. Compute each model's tokens-per-parameter ratio, and state by what factor GPT-3's training ratio falls short of Chinchilla's compute-optimal ratio.

--- Cycle 1 ---
Thought: First I need Chinchilla's tokens-per-parameter ratio.
Action: calculator(70 * 1.4)
Observation: 98.0

--- Cycle 2 ---
Thought: Now I can compute GPT-3's ratio.
Action: calculator(300 * 175)
Observation: 52500

--- Cycle 3 ---
Thought: I now have the ratio.
Final answer: 52500 / 98.0 = 53.3

Final answer: 52500 / 98.0 = 53.3
tokens_used=1405 latency_sec=17.03 cycles=3


/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



judge_score=0 (The candidate answer uses incorrect numbers and does not address the given problem at all.)


### B3. Plan → Solve → Verify chain

Composes patterns already built: one call writes a short plan, one call solves the problem following that plan (ending with a `Final answer:` line), and the self-critique machinery from A2 (`CRITIQUE_PROMPT_TEMPLATE` / `REVISE_PROMPT_TEMPLATE`) checks and revises that final answer.


In [39]:
PLAN_PROMPT_TEMPLATE = (
    "Question: {query}\n\n"
    "Write a short step-by-step PLAN (not the answer itself) for how to solve this question. "
    "List 2-4 concrete steps.\n"
    "Plan:"
)

SOLVE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Plan:\n{plan}\n\n"
    "Now follow the plan above to solve the question. End with a line in exactly this format:\n"
    "Final answer: <your answer>\n\n"
    "Solution:"
)

psv_trace = []

def pattern_plan_solve_verify(query):
    plan_prompt = PLAN_PROMPT_TEMPLATE.format(query=query)
    plan, p1, g1, l1 = call_generator_raw(plan_prompt, stop_strings=["\nQuestion:"])

    solve_prompt = SOLVE_PROMPT_TEMPLATE.format(query=query, plan=plan)
    solution, p2, g2, l2 = call_generator_raw(solve_prompt, stop_strings=["\nQuestion:"])
    solve_final = extract_final_answer(solution)

    critique_prompt = CRITIQUE_PROMPT_TEMPLATE.format(query=query, answer=solve_final)
    critique, p3, g3, l3 = call_generator_raw(critique_prompt, stop_strings=["### Question:", "\nQuestion:"])

    revise_prompt = REVISE_PROMPT_TEMPLATE.format(query=query, answer=solve_final, critique=critique)
    verified, p4, g4, l4 = call_generator_raw(revise_prompt, stop_strings=["### Question:", "\nQuestion:"])

    psv_trace.append({
        "query": query, "plan": plan, "solution": solution, "solve_final_answer": solve_final,
        "critique": critique, "verified_answer": verified,
    })
    tokens_used = (p1 + g1) + (p2 + g2) + (p3 + g3) + (p4 + g4)
    latency = l1 + l2 + l3 + l4
    return verified, tokens_used, latency


Plan, solution, and verified answer for one problem (Problem 1):


In [40]:
_ = pattern_plan_solve_verify(PROBLEMS_B[0]["question"])
t = psv_trace[-1]
print("Plan:\n", t["plan"], "\n")
print("Solution:\n", t["solution"], "\n")
print("Solve-stage final answer:", t["solve_final_answer"], "\n")
print("Critique:\n", t["critique"], "\n")
print("Verified answer:\n", t["verified_answer"])


Plan:
 1. Apply LoRA to the q_proj and v_proj layers in every layer. 2. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 3. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 4. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 5. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 6. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 7. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 8. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 9. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 10. 

Solution:
 1. Apply LoRA to the q_proj and v_proj layers in every layer. 2. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 3. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 4. Apply the LoRA adapter to the q_proj and v_proj layers in every layer. 5. Apply the LoRA adapter to the q_pro

Run on both problems and add a row to the B1 table:


In [41]:
for p in PROBLEMS_B:
    answer, tokens_used, latency = pattern_plan_solve_verify(p["question"])
    score, reason, _, _, _ = judge(p["question"], p["known_answer"], answer)
    b1_records.append({
        "problem_id": p["id"], "pattern": "plan_solve_verify", "query": p["question"],
        "answer": answer, "tokens_used": tokens_used, "latency_sec": latency,
        "judge_score": score, "judge_reason": reason,
    })
    print(f"[plan_solve_verify ] problem={p['id']} score={score} tokens={tokens_used:>4}")

b1_table = build_b1_table(b1_records, ["zero_shot", "chain_of_thought", "self_consistency", "plan_solve_verify"])
b1_table

results_partB = pd.DataFrame(b1_records)
results_partB.to_csv("results_partB.csv", index=False)
print(f"Saved {len(results_partB)} rows to results_partB.csv")
b1_table


[plan_solve_verify ] problem=1 score=0 tokens=2072
[plan_solve_verify ] problem=2 score=1 tokens=2016
Saved 8 rows to results_partB.csv


,problem_1_score,problem_2_score,avg_tokens
zero_shot,0.0,0.0,284.5
chain_of_thought,0.0,0.0,327.0
self_consistency,0.0,0.0,1571.5
plan_solve_verify,0.0,1.0,2044.0


### Part B — draft verdict: did the plan-solve-verify chain beat the best single pattern?


In [42]:
single_patterns = ["zero_shot", "chain_of_thought", "self_consistency"]
single_avg_scores = b1_table.loc[single_patterns, ["problem_1_score", "problem_2_score"]].mean(axis=1)
best_single_name = single_avg_scores.idxmax()
best_single_score = single_avg_scores.loc[best_single_name]
best_single_tokens = b1_table.loc[best_single_name, "avg_tokens"]

psv_score = b1_table.loc["plan_solve_verify", ["problem_1_score", "problem_2_score"]].mean()
psv_tokens = b1_table.loc["plan_solve_verify", "avg_tokens"]

if psv_score > best_single_score:
    comparison = f"the plan-solve-verify chain improved the average score from {best_single_score:.2f} ({best_single_name}) to {psv_score:.2f}"
    judgement = "justifies" if (psv_score - best_single_score) >= 0.3 else "only marginally justifies"
elif psv_score == best_single_score:
    comparison = f"the plan-solve-verify chain matched the best single pattern's average score of {best_single_score:.2f} ({best_single_name})"
    judgement = "does not justify"
else:
    comparison = f"the plan-solve-verify chain scored lower on average ({psv_score:.2f} vs {best_single_score:.2f} for {best_single_name})"
    judgement = "does not justify"

draft_b3 = (
    "DRAFT -- review before submission: "
    f"{comparison}, using {psv_tokens:.0f} tokens on average versus {best_single_tokens:.0f} for the best "
    f"single pattern ({psv_tokens / best_single_tokens:.1f}x). This {judgement} the extra token cost."
)
print(draft_b3)


DRAFT -- review before submission: the plan-solve-verify chain improved the average score from 0.00 (zero_shot) to 0.50, using 2044 tokens on average versus 284 for the best single pattern (7.2x). This justifies the extra token cost.


## Part C — Prompt router

Routes each query to the cheapest pattern likely to answer it well:

| Query type | Route to |
|---|---|
| Simple factual question | Zero-shot (A0) |
| Structured extraction | Structured output (A3) |
| Multi-step reasoning | Chain-of-thought (B1) |
| Needs a calculation or tool | ReAct (B2) |
| High-stakes, needs checking | Plan -> Solve -> Verify (B3) |

The classifier runs on the same fine-tuned SmolLM2 generator used throughout Parts A/B (not the judge model), to keep routing cheap -- the whole point of a router is to avoid paying for an expensive call before even picking a pattern.


In [43]:
CLASSIFIER_LABELS = ["factual", "structured", "reasoning", "tool", "high_stakes"]

ROUTE_NAMES = {
    "factual": "zero_shot",
    "structured": "structured",
    "reasoning": "chain_of_thought",
    "tool": "react",
    "high_stakes": "plan_solve_verify",
}

print("Labels -> routes:")
for label, route_name in ROUTE_NAMES.items():
    print(f"  {label:12s} -> {route_name}")


Labels -> routes:
  factual      -> zero_shot
  structured   -> structured
  reasoning    -> chain_of_thought
  tool         -> react
  high_stakes  -> plan_solve_verify


### C1. Query classifier

A zero-shot prompt that must return exactly one of the 5 labels above, with no explanation.


In [44]:
CLASSIFIER_PROMPT_TEMPLATE = (
    "Classify the question below into exactly one category. Respond with ONLY the category "
    "name, nothing else.\n\n"
    "Categories:\n"
    "factual - a simple factual question with a single direct answer\n"
    "structured - asks to extract or return structured fields (e.g. a claim with its source and evidence)\n"
    "reasoning - needs multiple reasoning steps to reach an answer, without needing precise arithmetic\n"
    "tool - needs an exact numeric calculation\n"
    "high_stakes - the answer needs to be carefully checked/verified before being trusted\n\n"
    "Question: {query}\n"
    "Category:"
)

print(CLASSIFIER_PROMPT_TEMPLATE)

def _normalize_label(text):
    label = text.strip().split()[0].lower().strip(".,:;\"'") if text.strip() else ""
    if label in CLASSIFIER_LABELS:
        return label
    lowered = text.lower()
    for cand in CLASSIFIER_LABELS:
        if cand in lowered:
            return cand
    return label  # may be invalid; caller treats as a miss

def classify_query(query, prompt_template=None):
    prompt = (prompt_template or CLASSIFIER_PROMPT_TEMPLATE).format(query=query)
    text, p_tok, g_tok, lat = call_generator_raw(prompt, stop_strings=["\nQuestion:", "\n\n"])
    return _normalize_label(text), p_tok + g_tok, lat


Classify the question below into exactly one category. Respond with ONLY the category name, nothing else.

Categories:
factual - a simple factual question with a single direct answer
structured - asks to extract or return structured fields (e.g. a claim with its source and evidence)
reasoning - needs multiple reasoning steps to reach an answer, without needing precise arithmetic
tool - needs an exact numeric calculation
high_stakes - the answer needs to be carefully checked/verified before being trusted

Question: {query}
Category:


Vijay's confirmation: the 5 draft test queries and gold labels above are confirmed as-is, no edits.


In [45]:
CLASSIFIER_TEST_SET = [
    {
        "query": "What is the role of RMSNorm in a transformer like SmolLM2?",
        "gold_label": "factual",
    },
    {
        "query": (
            "What technique does QLoRA use to quantize the base model weights, and which "
            "paper introduces it? Return the source paper, the claim, and the evidence."
        ),
        "gold_label": "structured",
    },
    {
        "query": PROBLEMS_B[0]["question"],  # LoRA trainable-parameter count
        "gold_label": "reasoning",
    },
    {
        "query": PROBLEMS_B[1]["question"],  # Chinchilla vs GPT-3 ratio -- same question B2's ReAct runs on
        "gold_label": "tool",
    },
    {
        "query": (
            "Why is RLHF's alignment of a language model's outputs with human preferences for "
            "helpfulness, honesty, and harmlessness especially critical to verify correctly "
            "before deployment?"
        ),
        "gold_label": "high_stakes",
    },
]

for item in CLASSIFIER_TEST_SET:
    print(f"[{item['gold_label']:12s}] {item['query'][:90]}")


[factual     ] What is the role of RMSNorm in a transformer like SmolLM2?
[structured  ] What technique does QLoRA use to quantize the base model weights, and which paper introduc
[reasoning   ] SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads (grouped-query att
[tool        ] Chinchilla's own compute-optimal training used 1.4 trillion tokens for its 70-billion-para
[high_stakes ] Why is RLHF's alignment of a language model's outputs with human preferences for helpfulne


In [46]:
def test_classifier(prompt_template, label):
    results = []
    for item in CLASSIFIER_TEST_SET:
        pred, tok, lat = classify_query(item["query"], prompt_template=prompt_template)
        results.append({**item, "predicted_label": pred, "correct": pred == item["gold_label"]})
    accuracy = sum(r["correct"] for r in results) / len(results)
    n_correct = sum(r["correct"] for r in results)
    print(f"Classifier {label} accuracy: {accuracy:.2f} ({n_correct}/5)")
    for r in results:
        mark = "OK" if r["correct"] else "WRONG"
        print(f"  gold={r['gold_label']:12s} pred={r['predicted_label']:12s} {mark:5s} -- {r['query'][:70]}")
    return accuracy, results

accuracy_v1, classifier_v1_results = test_classifier(CLASSIFIER_PROMPT_TEMPLATE, "v1")


Classifier v1 accuracy: 0.00 (0/5)
  gold=factual      pred=structured   WRONG -- What is the role of RMSNorm in a transformer like SmolLM2?
  gold=structured   pred=factual      WRONG -- What technique does QLoRA use to quantize the base model weights, and 
  gold=reasoning    pred=structured   WRONG -- SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value head
  gold=tool         pred=factual      WRONG -- Chinchilla's own compute-optimal training used 1.4 trillion tokens for
  gold=high_stakes  pred=reasoning    WRONG -- Why is RLHF's alignment of a language model's outputs with human prefe


If v1 scores below 4/5, an improved prompt is tried and both versions are shown (per the brief); if v1 already clears the bar, v2 is skipped.

**Note on this run:** the first labs run of this cell found v1=0.20 and v2=0.00 (v2 was *worse*), but an earlier version of the code below unconditionally switched the router onto v2 regardless. Since v2 never once predicted a valid label, every one of the 10 benchmark queries silently fell back to zero_shot in C2 -- not a classifier failure by itself, but a real bug in how the fallback was chosen. Fixed to use whichever of v1/v2 actually scores higher; cells below (and C3, which never finished running that time) need a fresh run.


In [47]:
CLASSIFIER_PROMPT_TEMPLATE_V2 = (
    "Classify the question into exactly one category. Respond with ONLY the category word, "
    "nothing else -- no explanation, no punctuation.\n\n"
    "Examples:\n"
    "Question: What year was the transformer architecture introduced?\n"
    "Category: factual\n\n"
    "Question: Extract the claim and its source paper about dynamic masking being used instead of static masking.\n"
    "Category: structured\n\n"
    "Question: If a 10-billion-parameter model needs 200 billion compute-optimal training tokens, "
    "how many tokens would a 40-billion-parameter model need under the same compute-optimal rule?\n"
    "Category: reasoning\n\n"
    "Question: A model has 12 layers with 64 parameters each. What is 12 times 64?\n"
    "Category: tool\n\n"
    "Question: Why must a medical diagnosis model's output be double-checked before being trusted?\n"
    "Category: high_stakes\n\n"
    "Question: {query}\n"
    "Category:"
)

if accuracy_v1 < 0.8:
    print(f"v1 accuracy {accuracy_v1:.2f} is below the 4/5 bar -- trying an improved prompt (v2).\n")
    print(CLASSIFIER_PROMPT_TEMPLATE_V2)
    print()
    accuracy_v2, classifier_v2_results = test_classifier(CLASSIFIER_PROMPT_TEMPLATE_V2, "v2")

    # Use whichever of v1/v2 actually scores higher on the labelled set -- v2 is an
    # attempted improvement, not a guaranteed one, and the router should not be
    # switched onto a prompt that empirically regressed.
    if accuracy_v2 > accuracy_v1:
        CLASSIFIER_PROMPT_FINAL = CLASSIFIER_PROMPT_TEMPLATE_V2
        classifier_accuracy_final = accuracy_v2
        print(f"v2 ({accuracy_v2:.2f}) improved on v1 ({accuracy_v1:.2f}) -- using v2 for the router.")
    else:
        CLASSIFIER_PROMPT_FINAL = CLASSIFIER_PROMPT_TEMPLATE
        classifier_accuracy_final = accuracy_v1
        print(
            f"v2 ({accuracy_v2:.2f}) did not improve on v1 ({accuracy_v1:.2f}) -- keeping v1 for the router. "
            "Both versions are shown above as the brief asks, but routing on a prompt that scored worse "
            "on the labelled set would make the router strictly worse for no reason."
        )
else:
    print(f"v1 accuracy {accuracy_v1:.2f} meets the 4/5 bar -- no improvement needed, v2 not used.")
    CLASSIFIER_PROMPT_FINAL = CLASSIFIER_PROMPT_TEMPLATE
    classifier_accuracy_final = accuracy_v1

final_version = "v2" if CLASSIFIER_PROMPT_FINAL == CLASSIFIER_PROMPT_TEMPLATE_V2 else "v1"
print(f"\nFinal classifier prompt in use for the router: {final_version} (accuracy {classifier_accuracy_final:.2f})")


v1 accuracy 0.00 is below the 4/5 bar -- trying an improved prompt (v2).

Classify the question into exactly one category. Respond with ONLY the category word, nothing else -- no explanation, no punctuation.

Examples:
Question: What year was the transformer architecture introduced?
Category: factual

Question: Extract the claim and its source paper about dynamic masking being used instead of static masking.
Category: structured

Question: If a 10-billion-parameter model needs 200 billion compute-optimal training tokens, how many tokens would a 40-billion-parameter model need under the same compute-optimal rule?
Category: reasoning

Question: A model has 12 layers with 64 parameters each. What is 12 times 64?
Category: tool

Question: Why must a medical diagnosis model's output be double-checked before being trusted?
Category: high_stakes

Question: {query}
Category:

Classifier v2 accuracy: 0.20 (1/5)
  gold=factual      pred=high_stakes  WRONG -- What is the role of RMSNorm in a tran

### C2. Router


In [48]:
ROUTE_FUNCS = {
    "zero_shot": pattern_zero_shot,
    "structured": pattern_structured,
    "chain_of_thought": pattern_cot,
    "react": pattern_react,
    "plan_solve_verify": pattern_plan_solve_verify,
}

route_trace = []

def route(query):
    label, cls_tokens, cls_latency = classify_query(query, prompt_template=CLASSIFIER_PROMPT_FINAL)
    is_fallback = label not in ROUTE_NAMES  # classifier returned something other than the 5 valid labels
    route_name = ROUTE_NAMES.get(label, "zero_shot")  # fall back to the cheapest pattern on an invalid label

    pattern_fn = ROUTE_FUNCS[route_name]
    raw_answer, pattern_tokens, pattern_latency = pattern_fn(query)
    answer = extract_final_answer(raw_answer) if route_name == "chain_of_thought" else raw_answer

    tokens_used = cls_tokens + pattern_tokens
    latency_sec = cls_latency + pattern_latency
    route_trace.append({
        "query": query, "classifier_label": label, "route": route_name, "fallback": is_fallback,
    })
    return answer, route_name, tokens_used, latency_sec


In [49]:
c2_records = []
for q in benchmark:
    query, known = q["instruction"], q["response"]
    answer, route_name, tokens_used, latency_sec = route(query)
    classifier_label = route_trace[-1]["classifier_label"]
    was_fallback = route_trace[-1]["fallback"]
    score, reason, judge_attempts, judge_tokens, judge_latency = judge(query, known, answer)
    c2_records.append({
        "source_idx": q["source_idx"], "query": query,
        "classifier_label": classifier_label, "fallback": was_fallback, "route": route_name,
        "judge_score": score, "judge_reason": reason,
        "tokens_used": tokens_used, "latency_sec": latency_sec,
    })
    fallback_note = " (fallback -- invalid label)" if was_fallback else ""
    print(f"idx={q['source_idx']:>2} label={classifier_label:12s} route={route_name:17s}{fallback_note} score={score} tokens={tokens_used:>4}")

results_partC = pd.DataFrame(c2_records)
results_partC["judge_score"] = pd.to_numeric(results_partC["judge_score"], errors="coerce")
results_partC.to_csv("results_partC.csv", index=False)

n_fallback = results_partC["fallback"].sum()
print(f"\nSaved {len(results_partC)} rows to results_partC.csv")
print(f"{n_fallback}/{len(results_partC)} queries fell back to zero_shot because the classifier returned an invalid label.")
results_partC[["source_idx", "query", "classifier_label", "route", "judge_score", "tokens_used"]]

/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


idx=27 label=high_stakes  route=plan_solve_verify score=0 tokens=1994
idx=30 label=high_stakes  route=plan_solve_verify score=0 tokens=1995
idx=25 label=high_stakes  route=plan_solve_verify score=0 tokens=1989
idx=57 label=high_stakes  route=plan_solve_verify score=0 tokens=2015
idx= 8 label=high_stakes  route=plan_solve_verify score=1 tokens=2009
idx= 4 label=high_stakes  route=plan_solve_verify score=0 tokens=1990
idx=14 label=language     route=zero_shot         (fallback -- invalid label) score=0 tokens= 286
idx=51 label=high_stakes  route=plan_solve_verify score=0 tokens=1999
idx= 3 label=data         route=zero_shot         (fallback -- invalid label) score=0 tokens= 298
idx=53 label=comparison   route=zero_shot         (fallback -- invalid label) score=1 tokens= 305

Saved 10 rows to results_partC.csv
3/10 queries fell back to zero_shot because the classifier returned an invalid label.


,source_idx,query,classifier_label,route,judge_score,tokens_used
0,27,How much memory does double quantization save ...,high_stakes,plan_solve_verify,0,1994
1,30,What is prefix-tuning and how many parameters ...,high_stakes,plan_solve_verify,0,1995
2,25,What are the three main innovations introduced...,high_stakes,plan_solve_verify,0,1989
3,57,How does PPO-ptx differ from the plain PPO mod...,high_stakes,plan_solve_verify,0,2015
4,8,How did the GPT-3 authors check whether benchm...,high_stakes,plan_solve_verify,1,2009
5,4,Outline how BERT is adapted to a downstream cl...,high_stakes,plan_solve_verify,0,1990
6,14,How does the LLaMA tokenizer handle numbers an...,language,zero_shot,0,286
7,51,How is the DPO training loss computed for a si...,high_stakes,plan_solve_verify,0,1999
8,3,Which corpora and model sizes were used for BE...,data,zero_shot,0,298
9,53,How does DPO compare with PPO-based RLHF in th...,comparison,zero_shot,1,305


### C3. Cost against quality

Reuses the zero-shot row from the A4 benchmark table (same judge prompt, same 10 queries) for a fair comparison.


In [50]:
always_zero_shot_row = {
    "avg_score": partA_table.loc["zero_shot", "avg_score"],
    "avg_tokens": partA_table.loc["zero_shot", "avg_tokens"],
    "avg_latency_sec": partA_table.loc["zero_shot", "avg_latency_sec"],
}
router_row = {
    "avg_score": results_partC["judge_score"].mean(),
    "avg_tokens": results_partC["tokens_used"].mean(),
    "avg_latency_sec": results_partC["latency_sec"].mean(),
}

partC_cost_table = pd.DataFrame({"always_zero_shot": always_zero_shot_row, "router": router_row}).T
partC_cost_table


,avg_score,avg_tokens,avg_latency_sec
always_zero_shot,0.1,109.6,1.838714
router,0.2,1488.0,22.958552


### Part C — draft ~100-word analysis


In [51]:
route_counts = results_partC["route"].value_counts()
most_common_route = route_counts.idxmax()

zs_by_idx = results_partA[results_partA["pattern"] == "zero_shot"].set_index("source_idx")
router_by_idx = results_partC.set_index("source_idx")

beat_idx = [
    idx for idx in router_by_idx.index
    if idx in zs_by_idx.index and router_by_idx.loc[idx, "judge_score"] > zs_by_idx.loc[idx, "judge_score"]
]

if beat_idx:
    beat_router_tokens = router_by_idx.loc[beat_idx, "tokens_used"].mean()
    beat_zs_tokens = zs_by_idx.loc[beat_idx, "tokens_used"].mean()
    token_ratio = beat_router_tokens / beat_zs_tokens
    beat_clause = (
        f"on {len(beat_idx)} of 10 queries, routing beat always-zero-shot, at {token_ratio:.1f}x the "
        "token cost on those queries -- "
        + ("worth it given the score gain" if token_ratio <= 3 else "a steep token cost for the gain")
    )
else:
    beat_clause = "on none of the 10 queries did routing score strictly higher than always-zero-shot"

single_pattern_scores = partA_table["avg_score"]
best_deploy_pattern = single_pattern_scores.idxmax()

draft_c3 = (
    "DRAFT -- review before submission: "
    f"the router chose {most_common_route} most often ({route_counts.loc[most_common_route]} of 10 queries). "
    f"{beat_clause}. "
    f"If forced to deploy a single pattern without routing, {best_deploy_pattern} would be the choice, since "
    f"it has the highest average judge score ({single_pattern_scores.loc[best_deploy_pattern]:.2f}) among "
    "Part A's single patterns in this run."
)
print(draft_c3)
print(f"\n(word count: {len(draft_c3.split())})")


DRAFT -- review before submission: the router chose plan_solve_verify most often (7 of 10 queries). on 1 of 10 queries, routing beat always-zero-shot, at 23.9x the token cost on those queries -- a steep token cost for the gain. If forced to deploy a single pattern without routing, instruction would be the choice, since it has the highest average judge score (0.20) among Part A's single patterns in this run.

(word count: 70)


## Export to HTML

Exports this notebook to `Assignment2A_Group172.html` via `nbconvert` -- the same file the Deliverables check below looks for. `nbconvert` reads whatever is currently saved on disk, so if running interactively, save the notebook (Ctrl/Cmd+S) right before running this cell for the most complete export. Note: since this cell runs *before* the Deliverables section, this particular export won't include the Deliverables table's own output -- re-run this cell (or the standalone `jupyter nbconvert` command in RUNBOOK.md) after a full run if needed.

In [52]:
import subprocess
import sys

NOTEBOOK_NAME = "Assignment2A_Group172.ipynb"
HTML_NAME = "Assignment2A_Group172.html"

result = subprocess.run(
    [sys.executable, "-m", "jupyter", "nbconvert", "--to", "html", NOTEBOOK_NAME],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
print(f"HTML export {'succeeded' if result.returncode == 0 else 'failed'}: {HTML_NAME}")



HTML export succeeded: Assignment2A_Group172.html


## Deliverables

A grounded check of every item the assignment brief's submission checklist asks for (files, then each Part A/B/C item), modelled on 1A's own deliverables check -- "present" here means the real artifact exists in this run's own namespace or on disk, not just that the cell above printed something that looked right.


In [53]:
from pathlib import Path

def file_check(path, note=""):
    p = Path(path)
    if p.is_dir():
        n = sum(1 for _ in p.glob("*"))
        return ("present" if n > 0 else "missing", f"{n} files" if not note else note)
    exists = p.exists()
    size_kb = p.stat().st_size / 1024 if exists else 0
    return ("present" if exists else "missing", f"{size_kb:.0f} KB" if not note else note)

def var_check(name, ok, note=""):
    return ("present" if ok else "missing", note)

rows = []

# --- Files ---
status, note = file_check("Assignment2A_Group172.ipynb")
rows.append(("Files", "Notebook (.ipynb) with all outputs visible", status, note))
status, note = file_check("Assignment2A_Group172.html")
rows.append(("Files", "HTML export of the same notebook", status, note if status == "present" else "export after this run finishes"))

# --- Part A ---
status, note = var_check("fewshot_examples", "fewshot_examples" in globals() and "INSTRUCTION_TEMPLATE" in globals(),
                          f"{len(fewshot_examples)} examples + instruction template" if "fewshot_examples" in globals() else "")
rows.append(("Part A", "A1: few-shot examples used and instruction template", status, note))

status, note = var_check("self_critique_trace", "self_critique_trace" in globals() and len(self_critique_trace) >= 2,
                          f"{len(self_critique_trace)} queries (first answer, critique, revised answer)" if "self_critique_trace" in globals() else "")
rows.append(("Part A", "A2: first answer, critique and revised answer for 2 queries", status, note))

if "structured_trace" in globals():
    valid_structured = [t for t in structured_trace if t.get("attempt_1_parsed") or t.get("attempt_2_parsed")]
    retried = [t for t in structured_trace if t.get("attempt_1_parsed") is None]
    ok_a3 = len(STRUCTURED_SCHEMA_FIELDS) > 0 and len(valid_structured) >= 3 and len(retried) >= 1
    note_a3 = f"schema + {len(valid_structured)} valid outputs + {len(retried)} parse failure(s) with retry"
else:
    ok_a3, note_a3 = False, ""
rows.append(("Part A", "A3: JSON schema, 3 valid outputs, one parse failure and its retry", "present" if ok_a3 else "missing", note_a3))

ok_a4 = all(n in globals() for n in ["JUDGE_MODEL_ID", "JUDGE_USER_TEMPLATE", "partA_table", "spot_check_sample", "draft_text"])
note_a4 = f"judge={JUDGE_MODEL_ID}" if "JUDGE_MODEL_ID" in globals() else ""
rows.append(("Part A", "A4: judge model name and judge prompt, benchmark table, spot-check result, self-critique verdict",
             "present" if ok_a4 else "missing", note_a4))

# --- Part B ---
status, note = var_check("PROBLEMS_B", "PROBLEMS_B" in globals() and len(PROBLEMS_B) == 2,
                          f"{len(PROBLEMS_B)} problems with known answers" if "PROBLEMS_B" in globals() else "")
rows.append(("Part B", "Your 2 multi-step problems with their known answers", status, note))

ok_b1 = all(n in globals() for n in ["cot_trace", "self_consistency_trace", "b1_table"])
note_b1 = f"{len(self_consistency_trace)} self-consistency votes, table rows: {list(b1_table.index)}" if ok_b1 else ""
rows.append(("Part B", "B1: chain-of-thought reasoning, the 5 self-consistency samples and vote, filled table",
             "present" if ok_b1 else "missing", note_b1))

ok_b2 = "calculator" in globals() and "react_trace" in globals() and len(react_trace) > 0
note_b2 = f"{len(react_trace[-1]['cycles'])} cycles, final answer: {react_trace[-1]['final_answer']!r}" if ok_b2 else ""
rows.append(("Part B", "B2: tool code and full ReAct trace with the final answer", "present" if ok_b2 else "missing", note_b2))

ok_b3 = "psv_trace" in globals() and len(psv_trace) > 0 and "plan_solve_verify" in b1_table.index and "draft_b3" in globals()
rows.append(("Part B", "B3: plan, solution and verified answer for one problem; chain row in the table; whether the chain was worth it",
             "present" if ok_b3 else "missing", "plan/solution/verified + table row + verdict" if ok_b3 else ""))

# --- Part C ---
ok_c1 = all(n in globals() for n in ["CLASSIFIER_PROMPT_TEMPLATE", "CLASSIFIER_PROMPT_TEMPLATE_V2", "accuracy_v1", "classifier_accuracy_final"])
note_c1 = f"v1={accuracy_v1:.2f}, final={classifier_accuracy_final:.2f}" if ok_c1 else ""
rows.append(("Part C", "C1: classifier prompt (both versions if refined) and accuracy on 5 labelled queries",
             "present" if ok_c1 else "missing", note_c1))

ok_c2 = "results_partC" in globals() and len(results_partC) == 10
note_c2 = f"{len(results_partC)} rows" if ok_c2 else ""
rows.append(("Part C", "C2: router results table for the 10 queries", "present" if ok_c2 else "missing", note_c2))

ok_c3 = "partC_cost_table" in globals() and "draft_c3" in globals()
rows.append(("Part C", "C3: router vs zero-shot table and 100-word analysis", "present" if ok_c3 else "missing",
             "cost table + draft analysis" if ok_c3 else ""))

deliverables_df = pd.DataFrame(rows, columns=["Section", "Deliverable", "Status", "Notes"])
n_missing = (deliverables_df["Status"] == "missing").sum()
print(f"{len(deliverables_df) - n_missing} / {len(deliverables_df)} deliverables present"
      + (f" -- {n_missing} missing, see table" if n_missing else " -- all present"))
deliverables_df


13 / 13 deliverables present -- all present


,Section,Deliverable,Status,Notes
0,Files,Notebook (.ipynb) with all outputs visible,present,176 KB
1,Files,HTML export of the same notebook,present,660 KB
2,Part A,A1: few-shot examples used and instruction tem...,present,3 examples + instruction template
3,Part A,"A2: first answer, critique and revised answer ...",present,"10 queries (first answer, critique, revised an..."
4,Part A,"A3: JSON schema, 3 valid outputs, one parse fa...",present,schema + 5 valid outputs + 5 parse failure(s) ...
5,Part A,"A4: judge model name and judge prompt, benchma...",present,judge=Qwen/Qwen2.5-7B-Instruct
6,Part B,Your 2 multi-step problems with their known an...,present,2 problems with known answers
7,Part B,"B1: chain-of-thought reasoning, the 5 self-con...",present,"2 self-consistency votes, table rows: ['zero_s..."
8,Part B,B2: tool code and full ReAct trace with the fi...,present,"3 cycles, final answer: '52500 / 98.0 = 53.3'"
9,Part B,"B3: plan, solution and verified answer for one...",present,plan/solution/verified + table row + verdict
